# Whisper Reference Pipeline

Demonstrates the **existing** Faster-Whisper / WhisperX pipeline.
This is the baseline before switching to the NVIDIA Parakeet backend.

Set `AUDIO_FILE` to a real recording to get real timing numbers.
Without a GPU, the pipeline will be slow — this notebook is for reference only.

In [ ]:
import sys
import time
from pathlib import Path

# ── Configuration ──────────────────────────────────────────────────────────
AUDIO_FILE = ""  # Set to absolute path of a .wav / .mp4 / .m4a file
# ──────────────────────────────────────────────────────────────────────────

PROJECT_ROOT = Path(".").resolve().parent
sys.path.insert(0, str(PROJECT_ROOT))

import torch
print(f"PyTorch:  {torch.__version__}")
print(f"CUDA:     {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU:      {torch.cuda.get_device_name(0)}")
    print(f"VRAM:     {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
from transcript_engine.config.settings import Settings, PipelineConfig
from transcript_engine.model_registry.registry import ModelRegistry
from transcript_engine.pipeline.orchestrator import Pipeline
from transcript_engine.audio.preprocessor import AudioPreprocessor

settings = Settings()
config = PipelineConfig()

# Ensure WhisperX backend (default)
import os
os.environ["TE_ASR_BACKEND"] = "whisper"

print(f"Model:          {config.transcription.model_id}")
print(f"Device:         {config.transcription.device}")
print(f"Compute type:   {config.transcription.compute_type}")
print(f"Diarization:    {config.diarization.enabled}")

In [ ]:
if not AUDIO_FILE or not Path(AUDIO_FILE).exists():
    print("No AUDIO_FILE set — skipping pipeline run.")
    print("Set AUDIO_FILE above to a real .wav / .mp4 / .m4a file.")
else:
    registry = ModelRegistry(settings)
    pipeline = Pipeline(config, registry)

    t0 = time.monotonic()
    result = pipeline.run(Path(AUDIO_FILE))
    elapsed = time.monotonic() - t0

    audio_min = result.audio.duration / 60
    wall_min = elapsed / 60
    rtf = result.audio.duration / elapsed if elapsed > 0 else 0

    print(f"Audio:          {audio_min:.1f} min")
    print(f"Wall time:      {wall_min:.1f} min")
    print(f"RTF:            {rtf:.1f}x")
    print(f"Words:          {result.transcript.word_count}")
    print(f"Speakers:       {len(result.transcript.speakers)}")
    print()
    print(result.transcript.to_markdown()[:2000])